# Import Required Libraries
Import necessary data manipulation and model loading libraries such as `pandas`, `numpy`, `joblib`, and `os`.

In [7]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load Model and Dataset
Load the pre-trained XGBoost and LightGBM model bundle, extract the weights and label encoders, and load the unseen 2024 dataset into a DataFrame.

In [8]:
# File Paths
data_path = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.9 - Model Validation\Datasets\2024_dataset_complete.csv'
model_path = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.8 - Retail Price Ensemble Models\XGBoost + LightBGM\Models\xgb_lgbm_advanced_ensemble_optuna_model.joblib'

# 1. Load the Model Bundle
print('Loading Advanced Model Bundle...')
bundle = joblib.load(model_path)

model_xgb = bundle['xgb']
model_lgb = bundle['lgb']
features = bundle['features']
best_weight_lgb = bundle['weights']['lgb']
best_weight_xgb = bundle['weights']['xgb']
le_dict = bundle['label_encoders']

print(f"Loaded! LGBM Weight: {best_weight_lgb:.4f} | XGB Weight: {best_weight_xgb:.4f}")

# 2. Load the Dataset
print('Loading 2024 Data...')
df = pd.read_csv(data_path)

# Correct the exact string encodings matching the original pipeline
df['retail_market'] = df['retail_market'].str.title()
df['vegetable_type'] = df['vegetable_type'].str.upper()
df['vegetable_zone'] = df['vegetable_zone'].str.title()

print(f"Initial Dataset Shape: {df.shape}")

Loading Advanced Model Bundle...
Loaded! LGBM Weight: 0.5317 | XGB Weight: 0.4683
Loading 2024 Data...
Initial Dataset Shape: (8181, 15)


# Apply Feature Engineering
Reconstruct the structural timelines, physical lags, rolling moments, and origin-averaging weather features necessary for the models.

In [9]:
# Setup structural timeline
df['week_str'] = df['week'].astype(str).str.zfill(2)
df['Year_Week'] = df['year'].astype(str) + '_' + df['week_str']
df['week_num'] = pd.to_numeric(df['week'].astype(str).str.extract(r'(\d+)')[0])
df['week_sin'] = np.sin(2 * np.pi * df['week_num'] / 52)
df['week_cos'] = np.cos(2 * np.pi * df['week_num'] / 52)

# Regional Weather aggregation
regional_weather = (
    df.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
    .mean().reset_index()
    .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
)
df = pd.merge(df, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')

# Secure classification mapping for previously trained encoders
if 'seasonality' in le_dict:
    le_season = le_dict['seasonality']
    # Safely handle 'unseen' classes to avoid crashing
    df['season_enc'] = df['seasonality'].astype(str).apply(lambda x: x if x in le_season.classes_ else le_season.classes_[0])
    df['season_enc'] = le_season.transform(df['season_enc'])
else:
    df['season_enc'] = LabelEncoder().fit_transform(df['seasonality'].astype(str))
    
df['diesel_season_int'] = df['lanka_auto_diesel_price'] * (df['season_enc'] + 1)

# Critical: Time-Series Sorting
df = df.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num']).reset_index(drop=True)

# Build physical Lags
for col in ['retail_price', 'reg_rain', 'reg_temp']:
    for lag in [1, 2, 3, 4, 8]:
        df[f'{col}_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

for lag in [1, 2, 3, 4, 5, 6, 8]:
    df[f'mean_farmer_price_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

# Rolling Moments
df['retail_price_roll_4'] = df.groupby(['retail_market', 'vegetable_type'])['retail_price'].transform(lambda x: x.shift(1).rolling(4).mean())
grp = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
df['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
df['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
df['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
df['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

df['mean_farmer_price_filled'] = df['mean_farmer_price'].fillna(df['mean_farmer_price_lag_1'])
df['farmer_retail_spread_lag_1'] = df['retail_price_lag_1'] - df['mean_farmer_price_lag_1']

df['retail_price_momentum_1_4'] = df['retail_price_lag_1'] / (df['retail_price_lag_4'] + 1e-5)
df['farmer_price_momentum_1_4'] = df['mean_farmer_price_lag_1'] / (df['mean_farmer_price_lag_4'] + 1e-5)

# Purge initial drop gaps caused by 8-week lags
df_ready = df.dropna(subset=['retail_price_lag_8', 'mean_farmer_price_lag_8', 'farmer_price_roll_8', 'retail_price_momentum_1_4']).copy()

# Enforce Categorical
for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
    if col in le_dict:
        le = le_dict[col]
        classes = list(le.classes_)
        df_ready[col] = df_ready[col].apply(lambda x: x if x in classes else classes[0])
        df_ready[f'{col}_enc'] = le.transform(df_ready[col])

df_ready = df_ready.dropna(subset=features)
print(f"Data Feature Engineered successfully. Final Test Rows: {len(df_ready)}")

Data Feature Engineered successfully. Final Test Rows: 6877


# Section 4 & 5: Prediction and Metric Mapping

In [10]:
# Slice out strict validation target
X_train = df_ready[features]
Y_true_log = np.log1p(df_ready['retail_price'])
Y_true = df_ready['retail_price']

# Extract Raw Arrays
Pred_xgb_log = model_xgb.predict(X_train)
Pred_lgb_log = model_lgb.predict(X_train)

# Blend Weights using pre-trained constants
FINAL_PREDS_LOG = best_weight_xgb * Pred_xgb_log + best_weight_lgb * Pred_lgb_log

# Inverse back to Physical Price
FINAL_PREDS = np.expm1(FINAL_PREDS_LOG)

# Calculate Margin of Errors
df_ready['Ensemble_Log_Predict'] = FINAL_PREDS_LOG
df_ready['Ensemble_Price_Predict'] = FINAL_PREDS

df_ready['XGB_Log'] = Pred_xgb_log
df_ready['LGB_Log'] = Pred_lgb_log

df_ready['Error_Percentage'] = np.abs((df_ready['retail_price'] - df_ready['Ensemble_Price_Predict']) / df_ready['retail_price']) * 100
df_ready['Absolute_Error_LKR'] = np.abs(df_ready['retail_price'] - df_ready['Ensemble_Price_Predict'])

print(df_ready[['retail_market', 'vegetable_type', 'retail_price', 'Ensemble_Price_Predict', 'Error_Percentage']].head(20))

   retail_market vegetable_type  retail_price  Ensemble_Price_Predict  \
8   Anuradhapura       BEETROOT         504.0              359.571390   
9   Anuradhapura       BEETROOT         408.0              410.650126   
10  Anuradhapura       BEETROOT         384.0              375.472337   
11  Anuradhapura       BEETROOT         352.0              343.284577   
12  Anuradhapura       BEETROOT         336.0              282.525810   
13  Anuradhapura       BEETROOT         272.0              279.309817   
14  Anuradhapura       BEETROOT         272.0              252.292511   
15  Anuradhapura       BEETROOT         352.0              248.795436   
16  Anuradhapura       BEETROOT         304.0              279.574155   
17  Anuradhapura       BEETROOT         304.0              276.808076   
18  Anuradhapura       BEETROOT         328.0              285.382584   
19  Anuradhapura       BEETROOT         352.0              341.929251   
20  Anuradhapura       BEETROOT         448.0      

# Section 6: Export CSV

In [11]:
export_cols = [
    'year', 'week_num', 'retail_market', 'vegetable_type', 'retail_price', 
    'Ensemble_Price_Predict', 'Error_Percentage', 'Absolute_Error_LKR',
    'mean_farmer_price', 'reg_rain', 'reg_temp', 'Ensemble_Log_Predict',
    'XGB_Log', 'LGB_Log'
]

# Output to final processed directory
output_df = df_ready[export_cols].copy()
output_df = output_df.round(2)

export_path = r"C:\Users\Ranuga\Data Science Project\Final Complete Dataset\Model Building\2024_Predictions_vs_Actuals.csv"
output_df.to_csv(export_path, index=False)
print(f"Data successfully exported to: {export_path}")

Data successfully exported to: C:\Users\Ranuga\Data Science Project\Final Complete Dataset\Model Building\2024_Predictions_vs_Actuals.csv
